# GOAPified Plan-and-Execute

This notebook demonstrates how **LangGoap** replaces the LLM-generated text plans in
[LangGraph's Plan-and-Execute](https://langchain-ai.github.io/langgraph/tutorials/plan-and-execute/plan-and-execute/)
with formal A\* GOAP planning.

## Original vs GOAPified

| Aspect | Original LangGraph | GOAPified (LangGoap) |
|--------|-------------------|---------------------|
| Planning | LLM generates text plan (list of strings) | A\* generates formal plan with preconditions/effects |
| Plan quality | May hallucinate impossible steps | A\* guarantees plan is achievable |
| Execution | ReAct agent interprets each text step | Typed action functions with verified preconditions |
| Replanning | LLM re-prompted to generate new text plan | Observer detects deviation, A\* replans from current state |
| Cost | LLM call per plan + per step | A\* search (no LLM needed for planning) |

## Scenario

"What is the hometown of the 2024 Australian Open winner?"

The original LLM plan would generate steps like:
1. Search for 2024 Australian Open winner
2. Find the winner's hometown
3. Report the answer

In LangGoap, each step is a formal action with typed preconditions and effects.
The A\* planner discovers the same sequence automatically.

In [1]:
from typing import Any

from langgoap import ActionSpec, GoalSpec, GoapGraph, ReplanStrategy

## Define Plan-and-Execute Actions

Each step in the research pipeline becomes a GOAP action.

In [2]:
def search_web(ws: dict[str, Any]) -> dict[str, Any]:
    """Search the web for information."""
    query = ws.get("question", ws.get("task", ""))
    return {
        "has_search_results": True,
        "search_results": [
            {"content": f"Search result for '{query}': Jannik Sinner won the 2024 Australian Open."},
            {"content": "Sinner is from San Candido, South Tyrol, Italy."},
        ],
    }


def extract_facts(ws: dict[str, Any]) -> dict[str, Any]:
    """Extract structured facts from search results."""
    results = ws.get("search_results", [])
    facts = [r["content"] for r in results]
    return {
        "has_extracted_facts": True,
        "facts": facts,
        "winner": "Jannik Sinner",
        "hometown": "San Candido, South Tyrol, Italy",
    }


def compose_response(ws: dict[str, Any]) -> dict[str, Any]:
    """Compose a final response from extracted facts."""
    winner = ws.get("winner", "unknown")
    hometown = ws.get("hometown", "unknown")
    return {
        "response_ready": True,
        "response": f"The hometown of the 2024 Australian Open winner ({winner}) is {hometown}.",
    }


actions = [
    ActionSpec(
        name="search_web",
        preconditions={"has_task": True},
        effects={"has_search_results": True},
        cost=1.0,
        execute=search_web,
    ),
    ActionSpec(
        name="extract_facts",
        preconditions={"has_search_results": True},
        effects={"has_extracted_facts": True},
        cost=1.0,
        execute=extract_facts,
    ),
    ActionSpec(
        name="compose_response",
        preconditions={"has_extracted_facts": True},
        effects={"response_ready": True},
        cost=1.0,
        execute=compose_response,
    ),
]

## Formal Plan Discovery

The A\* planner discovers `search_web → extract_facts → compose_response`.
Unlike the original LLM plan, this is **verified** to be achievable.

In [3]:
result = GoapGraph(actions=actions).invoke(
    goal=GoalSpec(conditions={"response_ready": True}),
    world_state={
        "has_task": True,
        "task": "What is the hometown of the 2024 Australian Open winner?",
    },
)

print(f"Status: {result['status']}")
print(f"Response: {result['world_state']['response']}")
print()

# Show the verified action sequence
successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Action sequence: {' → '.join(successful)}")

Status: goal_achieved
Response: The hometown of the 2024 Australian Open winner (Jannik Sinner) is San Candido, South Tyrol, Italy.

Action sequence: search_web → extract_facts → compose_response


## Execution History

Every step produces an `ActionResult` with state snapshots before and after
execution, enabling full traceability.

In [4]:
for entry in result["execution_history"]:
    print(f"  {entry.action_name}: success={entry.success}")
    print(f"    State before keys: {sorted(entry.state_before.keys())}")
    print(f"    State after keys:  {sorted(entry.state_after.keys())}")
    print()

  search_web: success=True
    State before keys: ['has_task', 'task']
    State after keys:  ['has_search_results', 'has_task', 'search_results', 'task']

  extract_facts: success=True
    State before keys: ['has_search_results', 'has_task', 'search_results', 'task']
    State after keys:  ['facts', 'has_extracted_facts', 'has_search_results', 'has_task', 'hometown', 'search_results', 'task', 'winner']

  compose_response: success=True
    State before keys: ['facts', 'has_extracted_facts', 'has_search_results', 'has_task', 'hometown', 'search_results', 'task', 'winner']
    State after keys:  ['facts', 'has_extracted_facts', 'has_search_results', 'has_task', 'hometown', 'response', 'response_ready', 'search_results', 'task', 'winner']



## Replanning on Step Failure

When a step fails (e.g., search API timeout), the observer triggers
replanning from the current state. This replaces the original's LLM
re-prompting with formal replanning.

In [5]:
call_count = {"search": 0}


def flaky_search(ws: dict[str, Any]) -> dict[str, Any]:
    """Fails on first call, succeeds on retry."""
    call_count["search"] += 1
    if call_count["search"] == 1:
        raise RuntimeError("Search API timeout")
    return search_web(ws)


replan_actions = [
    ActionSpec(
        name="search_web",
        preconditions={"has_task": True},
        effects={"has_search_results": True},
        cost=1.0,
        execute=flaky_search,
    ),
    ActionSpec(
        name="extract_facts",
        preconditions={"has_search_results": True},
        effects={"has_extracted_facts": True},
        execute=extract_facts,
    ),
    ActionSpec(
        name="compose_response",
        preconditions={"has_extracted_facts": True},
        effects={"response_ready": True},
        execute=compose_response,
    ),
]

result = GoapGraph(actions=replan_actions).invoke(
    goal=GoalSpec(conditions={"response_ready": True}),
    world_state={"has_task": True},
)

print(f"Status: {result['status']}")
print(f"Replans: {result['replan_count']}")
print(f"Search calls: {call_count['search']}")
failures = [h for h in result["execution_history"] if not h.success]
print(f"Failed attempts: {len(failures)}")

Action 'search_web' failed: Search API timeout


Status: goal_achieved
Replans: 1
Search calls: 2
Failed attempts: 1


## NEVER Replan Strategy

With `ReplanStrategy.NEVER`, the system terminates immediately on failure
instead of attempting recovery.

In [6]:
def always_fails(ws: dict[str, Any]) -> dict[str, Any]:
    raise RuntimeError("permanent failure")


never_actions = [
    ActionSpec(
        name="search_web",
        preconditions={"has_task": True},
        effects={"has_search_results": True},
        execute=always_fails,
    ),
    ActionSpec(
        name="extract_facts",
        preconditions={"has_search_results": True},
        effects={"has_extracted_facts": True},
        execute=extract_facts,
    ),
    ActionSpec(
        name="compose_response",
        preconditions={"has_extracted_facts": True},
        effects={"response_ready": True},
        execute=compose_response,
    ),
]

result = GoapGraph(actions=never_actions).invoke(
    goal=GoalSpec(
        conditions={"response_ready": True},
        replan_strategy=ReplanStrategy.NEVER,
    ),
    world_state={"has_task": True},
)

print(f"Status: {result['status']}")
print(f"Replans: {result['replan_count']}")

Action 'search_web' failed: permanent failure


Status: failed
Replans: 0


## Longer Pipeline: 5-Step Research

The planner handles arbitrarily long pipelines. Here, a 5-step research
pipeline is discovered automatically by A\*.

In [7]:
def search(ws: dict[str, Any]) -> dict[str, Any]:
    return {"has_raw_data": True, "raw_data": ["fact1", "fact2", "fact3"]}


def verify(ws: dict[str, Any]) -> dict[str, Any]:
    data = ws.get("raw_data", [])
    return {"has_verified_data": True, "verified_data": data[:2]}


def analyze(ws: dict[str, Any]) -> dict[str, Any]:
    return {"has_analysis": True, "analysis": "Comprehensive analysis of verified facts."}


def draft(ws: dict[str, Any]) -> dict[str, Any]:
    analysis = ws.get("analysis", "")
    return {"has_draft": True, "draft": f"Draft report: {analysis}"}


def finalize(ws: dict[str, Any]) -> dict[str, Any]:
    draft_text = ws.get("draft", "")
    return {"report_complete": True, "final_report": f"[FINAL] {draft_text}"}


pipeline_actions = [
    ActionSpec(name="search", preconditions={"has_task": True}, effects={"has_raw_data": True}, execute=search),
    ActionSpec(name="verify", preconditions={"has_raw_data": True}, effects={"has_verified_data": True}, execute=verify),
    ActionSpec(name="analyze", preconditions={"has_verified_data": True}, effects={"has_analysis": True}, execute=analyze),
    ActionSpec(name="draft", preconditions={"has_analysis": True}, effects={"has_draft": True}, execute=draft),
    ActionSpec(name="finalize", preconditions={"has_draft": True}, effects={"report_complete": True}, execute=finalize),
]

result = GoapGraph(actions=pipeline_actions).invoke(
    goal=GoalSpec(conditions={"report_complete": True}),
    world_state={"has_task": True, "task": "Research AI planning"},
)

print(f"Status: {result['status']}")
print(f"Report: {result['world_state']['final_report']}")
successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Pipeline: {' → '.join(successful)}")

Status: goal_achieved
Report: [FINAL] Draft report: Comprehensive analysis of verified facts.
Pipeline: search → verify → analyze → draft → finalize
